# 03 · Recomendações

**Inferência batch:** o modelo `@champion` calcula o top-10 de **todos** os usuários de
uma vez e grava numa tabela. O app só faz um `SELECT` — rápido e barato. É assim que
muitos portais montam o "recomendado para você".

Depois, um **monitoramento** simples: estamos recomendando matérias variadas ou
sempre as mesmas?

In [ ]:
# Parâmetro do Job (rodando à mão, fica o valor padrão)
dbutils.widgets.text("schema", "news_recsys_dev")
SCHEMA = f"workspace.{dbutils.widgets.get('schema')}"
print(SCHEMA)

In [ ]:
import mlflow
import pandas as pd
from mlflow import MlflowClient

mlflow.set_registry_uri("databricks-uc")
MODELO = f"{SCHEMA}.news_recommender"

versao = MlflowClient().get_model_version_by_alias(MODELO, "champion").version
modelo = mlflow.pyfunc.load_model(f"models:/{MODELO}@champion")
print(f"usando {MODELO} versão {versao} (@champion)")

In [ ]:
usuarios = spark.table(f"{SCHEMA}.silver_users").toPandas()
materias = spark.table(f"{SCHEMA}.bronze_articles").toPandas()

recs = modelo.predict(usuarios[["user_id"]])
gold = recs.merge(materias[["url", "title", "section", "image_url"]], on="url").assign(
    model_version=str(versao), generated_at=pd.Timestamp.now()
)
spark.createDataFrame(gold).write.mode("overwrite").saveAsTable(f"{SCHEMA}.gold_recommendations")
print(len(gold), "recomendações para", gold["user_id"].nunique(), "usuários")
gold.head(10)[["user_id", "rank", "section", "title"]]

## Monitoramento

- **Cobertura:** que fração do catálogo aparece em alguma recomendação?
- **Editorias lidas × recomendadas:** o modelo segue o que as pessoas leem ou cria
  uma "bolha"?

In [ ]:
cliques = spark.table(f"{SCHEMA}.silver_clicks").toPandas()
lidas = cliques.merge(materias[["url", "section"]], on="url")["section"].value_counts(normalize=True)
recomendadas = gold["section"].value_counts(normalize=True)
cobertura = gold["url"].nunique() / len(materias)

spark.createDataFrame(pd.DataFrame([{
    "generated_at": pd.Timestamp.now(), "model_version": str(versao),
    "cobertura": cobertura, "usuarios": gold["user_id"].nunique(),
}])).write.mode("append").saveAsTable(f"{SCHEMA}.model_monitoring")

print(f"cobertura do catálogo: {cobertura:.0%}")
pd.DataFrame({"lidas": lidas, "recomendadas": recomendadas}).fillna(0).head(10).plot.barh(
    figsize=(8, 4), title="Editorias: lidas × recomendadas"
);

### O que percebemos?

- A inferência usa o **alias** `@champion`, nunca "a última versão": o quality gate
  protege o app.
- Cada linha da tabela gold diz **qual versão** do modelo a gerou.
- Monitorar não é só acurácia: cobertura e diversidade mostram se o modelo criou uma bolha.

Fim do pipeline. O app Streamlit lê `gold_recommendations`.